In [ ]:
import pandas as pd
import numpy as np

In [ ]:
matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')

In [ ]:
#Basic Exploration
print(matches.shape)
print(deliveries.shape)

print(matches.columns.tolist())
print(deliveries.columns.tolist())

print(matches.head())
print(deliveries.head())


print(matches.isnull().sum())
print(deliveries.isnull().sum())

In [ ]:
matches.describe()
deliveries.describe()

In [ ]:
#Total Matches won by Each Team (2008-2024)
matches['winner'].value_counts()

In [ ]:
#Merging Old team name with the new one

matches['winner'] = matches['winner'].replace({ 'Kings XI Punjab':'Punjab Kings',
    'Delhi Daredevils':'Delhi Capitals',
    'Deccan Chargers':'Sunrisers Hyderabad',
    'Rising Pune Supergiant':'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'})

matches['winner'].value_counts()

In [ ]:
#Cleaning and Making names of venues consistent

matches['venue'] = matches['venue'].str.split(',').str[0]
matches['venue'] = matches['venue'].replace({'M.Chinnaswamy Stadium':'M Chinnaswamy Stadium',
    'Punjab Cricket Association Stadium':'Maharaja Yadavindra Singh International Cricket Stadium',
    'Feroz Shah Kotla':'Arun Jaitley Stadium',
    'Sardar Patel Stadium':'Narendra Modi Stadium',
    'Zayed Cricket Stadium':'Sheikh Zayed Stadium'
    })
matches['venue'].unique()

In [ ]:
#Making Years of IPL seasons Consistent
matches['season'] = matches['season'].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'})

#Checking how many times toss winner team won the matches and percentage of matches won by teams who won the toss
toss_winners = matches[matches['toss_winner'] == matches['winner']]
toss_winners.shape
toss_win_count = toss_winners.shape[0]

#Omitting the matches with no Result
matches_clean = matches[matches['winner'].notna()]
matches_count = matches_clean.shape[0]

#Finding percentages of the match won by the teams who won the Toss

win_percentage = ((toss_win_count/matches_count)*100)
print(f"{win_percentage:.2f}%")

In [ ]:
#Finding Top 10 run scorer 
batter_runs = deliveries.groupby('batter')['batsman_runs'].sum()
batter_runs = batter_runs.sort_values(ascending=False)
top_10_batters = batter_runs.head(10).reset_index()
top_10_batters

In [ ]:
#Finding Batting Averages
matches_played_all = deliveries.groupby('batter')['match_id'].nunique().rename('matches_played')
batter_averages = top_10_batters.merge(matches_played_all,on = 'batter')
batter_averages = batter_averages[['batter','matches_played']]
batter_averages = batter_averages.merge(batter_runs,on = 'batter')
batter_averages['average'] = batter_averages['batsman_runs']/batter_averages['matches_played']

#sorting Batter on the basis of averages
batter_averages = batter_averages.sort_values('average',ascending=False)
batter_averages



In [ ]:
#Starting Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Visualizing top Batters with respect to average using Bar Chart
plt.bar(batter_averages['batter'],batter_averages['average'])
plt.title("Top 10 Batters With Respect To Batting Average")
plt.xlabel('batters')
plt.ylabel('Average')
plt.xticks(rotation=45,ha='right')
plt.tight_layout()
plt.savefig('../visuals/top_10_batters.png',bbox_inches='tight')
plt.show()

In [ ]:
#Finding top 10 Bowlers
bowler_wickets = deliveries.groupby('bowler')['is_wicket'].sum()
bowler_wickets = bowler_wickets.sort_values(ascending=False)
top_10_bowlers = bowler_wickets.head(10).reset_index()
top_10_bowlers

In [ ]:
#Creating a bar chart of Top Bowlers
plt.bar(top_10_bowlers['bowler'],top_10_bowlers['is_wicket'])
plt.title("Top 10 Bowlers With Respect To Wickets Taken")
plt.xlabel('Bowler')
plt.ylabel('Wickets')
plt.xticks(rotation = 45,ha ='right')
plt.tight_layout()
plt.savefig('../visuals/top_10_bowlers.png',bbox_inches='tight')
plt.show()

In [ ]:
#Total Matches Hosted by Each Venue (2008-2024)
hosted_each = matches['venue'].value_counts()
top_10_venues = matches['venue'].value_counts().head(10)
top_10_venues


In [ ]:
#Visualizing top 10 venues with respect to matches hosted

plt.barh(top_10_venues.index,top_10_venues.values)
plt.title("Top 10 Venues With Respect To Matches Hosted")
plt.tight_layout()
plt.savefig('../visuals/top_10_venues.png',bbox_inches='tight')
plt.show()

In [ ]:
#Venue and Toss Analysis
venue_and_toss = matches[['venue', 'toss_winner', 'toss_decision', 'winner']]
toss_venue = matches.groupby(['venue','toss_decision']).size().reset_index(name = 'count')
toss_wins_venue = venue_and_toss[venue_and_toss['toss_winner']==venue_and_toss['winner']]
toss_match_win = toss_wins_venue.groupby(['venue']).size().sort_values(ascending=False).head(10)
toss_match_win

In [ ]:
#Visualizing Toss and Venue Analysis
pivot_toss = pd.pivot_table(toss_venue,index = 'venue',columns='toss_decision',values = 'count',aggfunc='sum')
sns.heatmap(pivot_toss, fmt='g')
plt.title('Venue and Toss Analysis Visualization')
plt.tight_layout()
plt.savefig('../visuals/Venue and Toss Analysis.png',bbox_inches='tight')
plt.show()

In [ ]:
#Season wise Analysis

season_match_count = matches['season'].value_counts().reset_index(name='matches_played')
season_match_count['season'] = season_match_count['season'].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'})
season_match_count = season_match_count.sort_values('season')
season_match_count

In [ ]:
#visualizing matches played in each season
plt.plot(season_match_count['season'],season_match_count['matches_played'],marker = 'o')
plt.tight_layout()
plt.xticks(rotation=45,ha='right')
plt.title('IPL Matches Per Season')
plt.xlabel('Season')
plt.ylabel('Matches Played')
plt.savefig('../visuals/ipl_matches_per_season.png',bbox_inches='tight')
plt.show()

In [ ]:
#Checking the ratio between wins by runs and wins by wickets

win_by_runs =  matches['result'].value_counts()['runs']
win_by_wickets =  matches['result'].value_counts()['wickets']
tie_count =  matches['result'].value_counts()['tie']
nr_count =  matches['result'].value_counts()['no result']
total_matches = win_by_runs + win_by_wickets
wickets_pct = win_by_wickets/total_matches*100
runs_pct = win_by_runs/total_matches*100
print(f"Ratio between Matches won while Defending and while Chasing is {runs_pct:.0f}:{wickets_pct:.0f} ")

In [ ]:
#Visualizing the Ratio between wins by runs and wins by wickets

plt.pie([win_by_wickets,win_by_runs],labels = ['win_by_wickets','win_by_runs'],autopct='%1.1f%%')
plt.title("Match Results Distribution - Runs Vs Wickets")
plt.tight_layout()
plt.savefig('../visuals/matches_results.png',bbox_inches='tight')
plt.show()

In [ ]:
#Toss decisions Trends Over the Years

toss_trends = matches.groupby(['season','toss_decision']).size().reset_index()
toss_trends = toss_trends.rename(columns={0:'count'})
toss_trends

In [ ]:
#VisualIng toss trends

sns.lineplot(data = toss_trends,x= 'season',y= 'count',hue='toss_decision',marker = 'o')
plt.xticks(rotation = 45,ha = 'right')
plt.title('Toss Decision Trends Over The Years in IPL')
plt.xlabel('Seasons')
plt.ylabel('Decision Counts')
plt.tight_layout()
plt.savefig('../visuals/Toss Decision Trends Over The Years in IPL.png',bbox_inches='tight')
plt.show()